# CareerPilot AI — 2. Job Fetching

Pulls live postings from the Adzuna API and saves them to `job_dataset.json`.

**This is a standalone notebook.** It doesn't need `1_resume_parsing.ipynb` to have run first, and doesn't touch `careerpilot.db` at all — it only produces `job_dataset.json`, which `3_job_matching.ipynb` reads later.

**Run this once ahead of your demo, not live during it.** Re-run any time you want fresher listings — but you're on Trial Access, so don't hammer it while testing.


## Step 1 — Setup


In [4]:
!pip install requests --quiet
import json
import requests


## Step 2 — Skill tagging

A plain keyword list — the **same list** used in `1_resume_parsing.ipynb`'s Step 4, kept in sync by hand so job skills and profile skills use identical vocabulary for matching later. If you add a skill to one notebook, add it to the other too.


In [15]:
SKILLS = [
    "Python", "Java", "JavaScript", "SQL", "React", "FastAPI", "Flask",
    "Machine Learning", "Deep Learning", "AWS", "Docker", "Kubernetes",
    "Git", "spaCy", "MongoDB", "PostgreSQL", "TensorFlow", "PyTorch",
    "Node.js", "HTML", "CSS", "C++", "Data Analysis", "NLP",
]

def get_skills(text: str) -> list:
    # simple case-insensitive substring match — no spaCy dependency in this notebook
    text_lower = text.lower()
    return sorted({skill for skill in SKILLS if skill.lower() in text_lower})

print("Skill tagger ready.")


Skill tagger ready.


## Step 3 — Fetch from Adzuna

Searches a handful of terms, de-duplicates overlapping results, tags skills, and keeps the direct application link.


In [16]:
# --- Adzuna credentials ---
# Keep these out of anything you commit publicly (e.g. a public GitHub repo).
ADZUNA_APP_ID = "6abd39fa"
ADZUNA_APP_KEY = "1a615aa69c7325a614caa450b165077f"
ADZUNA_COUNTRY = "in"   # Adzuna also supports gb, us, au, ca, de, fr, and others
ADZUNA_SEARCH_TERMS = ["python developer", "data analyst", "software engineer", "machine learning"]
RESULTS_PER_TERM = 10

def fetch_adzuna_jobs(search_term: str, country: str = ADZUNA_COUNTRY, results: int = RESULTS_PER_TERM) -> list:
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/1"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "results_per_page": results,
        "what": search_term,
        "content-type": "application/json",
    }
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()   # fail loudly on a bad key / bad request
    return response.json().get("results", [])

def normalize_adzuna_job(raw: dict) -> dict:
    description = raw.get("description", "")
    return {
        "title": raw.get("title", "Untitled"),
        "company": raw.get("company", {}).get("display_name", "Unknown"),
        "location": raw.get("location", {}).get("display_name", ""),
        "skills": get_skills(description),
        "description": description,
        "apply_link": raw.get("redirect_url", ""),
    }

all_jobs = []
seen = set()

for term in ADZUNA_SEARCH_TERMS:
    raw_results = fetch_adzuna_jobs(term)
    for raw in raw_results:
        job = normalize_adzuna_job(raw)
        key = (job["title"], job["company"])
        if key not in seen:
            seen.add(key)
            all_jobs.append(job)

print(f"Fetched {len(all_jobs)} unique job postings.")


Fetched 34 unique job postings.


## Step 4 — Save to job_dataset.json


In [19]:
with open("job_dataset.json", "w") as f:
    json.dump(all_jobs, f, indent=2)

print("Saved job_dataset.json — ready for 3_job_matching.ipynb")


Saved job_dataset.json — ready for 3_job_matching.ipynb


In [18]:
#Run this cell to check api connection status
!curl -v --max-time 10 https://api.adzuna.com 2>&1 | head -30